# Radiographic model
In this script the foloowing are performed:
- Train-test-val split startified by patient
- Training of DL model
- Evaluation on val, test, ext validation at a follow-up level  

## Importing libraries

In [ ]:
import pandas as pd
from math import sqrt
import os
import random
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.svm import SVC
from sklearn.feature_selection import SelectFromModel
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, BayesianRidge
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
import sklearn.impute
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import seaborn as sb
import statistics
import seaborn as sn
from sklearn.model_selection import StratifiedKFold
from collections import Counter
import sklearn.ensemble
from statistics import mean
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from sklearn import metrics
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import KFold, StratifiedKFold, RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import random
import pandas as pd
import scipy.stats as stats
import os

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss

from sklearn.model_selection import GridSearchCV, KFold,RepeatedKFold
from sklearn.linear_model import Lasso
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.feature_selection import SelectKBest, f_classif, RFE

from sklearn import metrics
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from sklearn.neighbors import KNeighborsClassifier as KNN
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, KFold,RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# from sksurv.linear_model import CoxnetSurvivalAnalysis
# from sksurv.metrics import concordance_index_censored
# from sksurv.util import Surv

In [ ]:
#@title Set up the environment
import numpy as np
import sys
import pandas as pd
from collections import Counter, defaultdict
from IPython.display import Image
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dense,Flatten,Input,Lambda,GlobalAveragePooling2D,Dropout,Conv2D,BatchNormalization,MaxPooling2D,Input,Activation
from tensorflow.keras.models import Model,Sequential
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator,load_img,img_to_array,array_to_img
from tensorflow.keras.optimizers import Adam,RMSprop,SGD
from tensorflow.keras.applications.densenet import DenseNet169, preprocess_input
from tensorflow.keras.callbacks import ModelCheckpoint,EarlyStopping,ReduceLROnPlateau
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.decomposition import FastICA, PCA
from sklearn.metrics import accuracy_score, f1_score
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from tensorflow.keras.models import load_model
import numpy as np
from glob import glob
import matplotlib.pyplot as plt
import os
import math
from tensorflow.keras.regularizers import l2
import json
import pickle
import random

## Loading the data
the data must be divided into two folders:
- Class 1: contianing all failed rx
- Class 0: containing all not-failed rx

In [ ]:
#@title Set the path
train_path=r''
output_path = ''
if not os.path.exists(output_path): #create folder if it doesn't exist
    os.makedirs(output_path)

train_failed=os.path.join(train_path,'classe_1')
train_notfal=os.path.join(train_path,'classe_0')
print(len(os.listdir(train_failed)))
print(len(os.listdir(train_notfal)))

In [ ]:
SEED = 4224
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

In [ ]:
AP_image = os.path.join(train_notfal,'')
LAT_image = os.path.join(train_notfal,'')

es_AP = load_img(AP_image,target_size=(224,224))
es_LAT = load_img(LAT_image,target_size=(224,224))

In [ ]:
#Extract the files

fal_names = []
fal_code = []
nonfal_names = []
nonfal_code = []

for sample in os.listdir(train_failed):
    fal_code.append(sample.split('_')[0]) #in order to read only the unique code of the signal
    sample_name = os.path.join(train_failed, sample)
    fal_names.append(sample_name)

for sample in os.listdir(train_notfal):
    nonfal_code.append(sample.split('_')[0])
    sample_name = os.path.join(train_notfal, sample)
    nonfal_names.append(sample_name)

In [ ]:
#We create a dataframe to organize the data and their labels

labels_df=pd.DataFrame(columns=['img','label'])


### Train-test split
Stratified per patient

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Creiamo un DataFrame per organizzare immagini e label
labels_df = pd.DataFrame(columns=['img', 'label'])

# Popoliamo il DataFrame con le immagini e le label
i = 0
for name in fal_names:
    labels_df.loc[i] = pd.Series({'img': name, 'label': 'failed'})
    i += 1
for nf_name in nonfal_names:
    labels_df.loc[i] = pd.Series({'img': nf_name, 'label': 'not_failed'})
    i += 1

# Estrai gli ID paziente
labels_df['patient_id'] = labels_df['img'].str.extract(r'(O[1]-A-\d{4})')

# Raggruppiamo le label per paziente (tenendo conto di AP e LAT)
patient_labels = labels_df.groupby('patient_id')['label'].apply(lambda x: tuple(x))

# Ottenere gli ID unici dei pazienti
unique_patient_ids = patient_labels.index

In [ ]:
unique_patient_ids

In [ ]:
from sklearn.model_selection import train_test_split

# Conta il numero di immagini failed per paziente
patient_label_counts = labels_df.groupby('patient_id')['label'].value_counts().unstack(fill_value=0)

# Se manca una colonna (es. nessun "failed" in tutto il dataset), la aggiungo per sicurezza
for label in ['failed', 'not_failed']:
    if label not in patient_label_counts.columns:
        patient_label_counts[label] = 0

# Calcolo una "etichetta" sintetica per paziente = label con più immagini
patient_label_counts['majority_label'] = patient_label_counts[['failed', 'not_failed']].idxmax(axis=1)

# Ora possiamo fare stratificazione sui pazienti in base alla label dominante
train_patient_ids, test_patient_ids = train_test_split(
    patient_label_counts.index,
    test_size=0.2,
    stratify=patient_label_counts['majority_label'],
    random_state=42
)

# Assegna train/test nel dataframe
labels_df['split'] = labels_df['patient_id'].apply(
    lambda x: 'train' if x in train_patient_ids else 'test'
)


In [ ]:
# Filtra i dati di training
train_df = labels_df[labels_df['split'] == 'train']
unique_train_ids = train_df['patient_id'].unique()

# Conta le label per ogni paziente nel train
patient_label_counts_train = train_df.groupby('patient_id')['label'].value_counts().unstack(fill_value=0)

In [ ]:
for label in ['failed', 'not_failed']:
    if label not in patient_label_counts_train.columns:
        patient_label_counts_train[label] = 0

# Determina la label dominante per ogni paziente
patient_label_counts_train['majority_label'] = patient_label_counts_train[['failed', 'not_failed']].idxmax(axis=1)

In [ ]:
# Esegui lo split tra train e validation
train_patient_ids, val_patient_ids = train_test_split(
    patient_label_counts_train.index,
    test_size=0.2,
    stratify=patient_label_counts_train['majority_label'],
    random_state=42
)

# Assegna la split solo ai pazienti originariamente in train
labels_df['split'] = labels_df.apply(
    lambda row: 'val' if row['patient_id'] in val_patient_ids else (
        'train' if row['patient_id'] in train_patient_ids else row['split']
    ),
    axis=1
)

In [ ]:
len(val_patient_ids)

In [ ]:
# Conta il numero di immagini failed e not_failed per ogni split
split_counts = labels_df.groupby(['split', 'label']).size().unstack(fill_value=0)

# Stampa il risultato
print(split_counts)


## Adapting rx to Densenet model

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

seed = 4224
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)


In [ ]:
from tensorflow.keras import backend as K
os.environ['TF_DETERMINISTIC_OPS'] = '1'

In [ ]:
# Creiamo i DataFrame per train, val e test
learning_df = labels_df[labels_df['split'] == 'train']
val_df = labels_df[labels_df['split'] == 'val']
test_df = labels_df[labels_df['split'] == 'test']

In [ ]:
# Funzione per caricare e preprocessare le immagini
def load_and_preprocess_images(df):
    data_img = []
    valid_paths = []
    
    for img_path in df['img']:
        try:
            img = load_img(img_path, target_size=(224, 224))
            img_array = img_to_array(img)
            img_array = tf.keras.applications.densenet.preprocess_input(img_array)
            data_img.append(img_array)
            valid_paths.append(img_path)  # Salva i path validi
        except Exception as e:
            print(f"[SKIPPED] Immagine non valida: {img_path} -> {e}")
    
    return np.array(data_img), valid_paths

# Carichiamo le immagini per ogni set
x_train, valid_train_paths = load_and_preprocess_images(learning_df)
x_val, valid_val_paths = load_and_preprocess_images(val_df)
x_test, valid_test_paths = load_and_preprocess_images(test_df)

# Se vuoi aggiornare i DataFrame per riflettere solo i file validi:
learning_df = learning_df[learning_df['img'].isin(valid_train_paths)].reset_index(drop=True)
val_df = val_df[val_df['img'].isin(valid_val_paths)].reset_index(drop=True)
test_df = test_df[test_df['img'].isin(valid_test_paths)].reset_index(drop=True)

# Controlliamo le shape delle immagini caricate
print(f"x_train shape: {x_train.shape}")
print(f"x_val shape: {x_val.shape}")
print(f"x_test shape: {x_test.shape}")




In [ ]:
len(learning_df)

In [ ]:
labels=learning_df.label.to_list()
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(['failed','not_failed'])
train_labels = le.transform(labels)
y_train=to_categorical(train_labels,2)
y_train[:10],learning_df[:10]

In [ ]:
labels_val=val_df.label.to_list()
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(['failed','not_failed'])
val_labels = le.transform(labels_val)
y_val=to_categorical(val_labels,2)
y_val[:10],val_df[:10]

In [ ]:
labels_test=test_df.label.to_list()
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(['failed','not_failed'])
test_labels = le.transform(labels_test)
y_test=to_categorical(test_labels,2)
y_test[:10],test_df[:10]

In [ ]:
import re

def extract_patient_id_with_checkpoint(path):
    # Qui estrai O1-A-0128-12m dal path
    # Pattern: O1-A-<4 cifre>-<checkpoint>, esempio: O1-A-0128-12m
    match = re.search(r'(O1-A-\d{4}-\d+m)', path)
    if match:
        return match.group(1)
    else:
        return None  # o la vecchia patient_id se preferisci

val_df['patient_id'] = val_df['img'].apply(extract_patient_id_with_checkpoint)
train_df['patient_id'] = train_df['img'].apply(extract_patient_id_with_checkpoint)
test_df['patient_id'] = test_df['img'].apply(extract_patient_id_with_checkpoint)


In [ ]:
#@title Set index failed
if y_train[2,0] == 1:
  fail_pos = 0 #index where the failed label is positive
else:
  fail_pos = 1

In [ ]:
fail_pos

### Transfer learning DenseNet 169
The model was trained without using the class wieghts

In [ ]:
from tensorflow.keras.applications import DenseNet169

imagesize= (224, 224, 3)

densenet = DenseNet169(input_shape= imagesize, weights= 'imagenet', include_top= False)

In [ ]:
for layer in densenet.layers[:-180]:
    layer.trainable = False

print("Layer scongelati (trainable=True):")
for layer in densenet.layers[-180:]:
    layer.trainable = True
    print(layer.name)


In [ ]:
from tensorflow.keras.regularizers import l1

#@title Then we add the classification layer in output of the model
output=densenet.output

out=GlobalAveragePooling2D(name='globalavg')(output)
out = Dense(128, activation='relu')(out)
out=BatchNormalization()(out)
out=Dropout(0.3)(out)



pred=Dense(2,activation='softmax',name='prediction')(out)
model=Model(inputs=densenet.input,outputs=pred)
#model.summary()

In [ ]:
BATCH_SIZE=32
    # Data generators
datagen = ImageDataGenerator(
        rotation_range=30,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )
datagen_val=ImageDataGenerator() #note that is empty
traingen = datagen.flow(x_train, y_train, shuffle=True,
                                     batch_size=BATCH_SIZE)
validgen = datagen_val.flow(x_val, y_val, shuffle=True,
                                     batch_size=BATCH_SIZE)

In [ ]:
# Early Stopping = stop training when a monitored metric has stopped improving
# patience = Number of epochs with no improvement after which training will be stopped
# monitor = Quantity to be monitored. Defaults to "val_loss".
# min_delta = Minimum change in the monitored quantity to qualify as an improvement
# mode = when the training will stop
es = EarlyStopping(patience=25, monitor='val_loss', mode='auto', verbose=1, min_delta=0,restore_best_weights=False)
pl = ReduceLROnPlateau(monitor='val_loss', patience=10, factor=0.1, verbose=1, min_lr=1e-11)
callbacks_list = [es,pl]

In [ ]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
# Definisci l'ottimizzatore
optimizer = Adam(learning_rate=0.00009)

# Numero di classi per il modello
output_classes = 2  # Cambia questo valore in base alle tue classi

# Definizione delle metriche personalizzate
f1_metric = tf.keras.metrics.F1Score(average='macro', name='f1_score')
#balanced_acc = BalancedAccuracy()

# Compila il modello con le metriche personalizzate
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=[f1_metric,'accuracy'])


#### Training the model

In [ ]:
epochs=200
totalTrain=len(traingen)
totalValid=len(validgen)
history = model.fit(
    traingen,
    verbose=1,
    steps_per_epoch=totalTrain,
    validation_data=validgen,
    validation_steps=totalValid,
    epochs=epochs,
    callbacks=callbacks_list
    #class_weight=class_weights_dict
    )

In [ ]:
model.save('DenseNet_128_180_64.keras')  # Salva il modello in formato HDF5

In [ ]:
model=load_model('DenseNet_128_180.keras')  # Salva il modello in formato HDF5

### Evaluation of model

In [ ]:
def compute_tnr_tpr (model, data, label, fail_pos):
    '''Compute Tnr and Tpr from data

    Parameters:
        model: (array) classifier model
        data: (array) data which we want to compute predictions
        label: (array) label of the data
        fail_pos: (bool) index where the label "fail" is positive: indice della colonna in cui è presente 1 dei falliti
    -------------
    Return:
        tnr: (float) total negative rate
        tpr: (float) total positive rate'''
    if (len(data.shape) > 2): #deep models
      y_pred = model.predict(data)
      tn, fp, fn, tp = sklearn.metrics.confusion_matrix(label[:,fail_pos],(y_pred[:,fail_pos]>0.5)*1).ravel()
      #not formally corrected because softmax has not a threshold of 0.5 but the probability of the predicted will always be over 0.5 for two classes
      tnr = tn / (tn + fp) #specificity
      tpr = tp / (tp + fn) #sensitivity
      fnr = fn / (fn + tp)
      fpr = fp / (fp + tn)
    else: #ml models
      y_pred = model.predict(data)
      tn, fp, fn, tp = sklearn.metrics.confusion_matrix(label, y_pred).ravel()
      tnr = tn / (tn + fp)
      tpr = tp / (tp + fn)
      fnr = fn / (fn + tp)
      fpr = fp / (fp + tn)

    return tn, tp, fn, fp, tnr, tpr, fnr, fpr


def plot_loss_figure (history_tuned, fold):
    '''Plot loss and accuracy over epochs and save them using fold index

    Parameters:
        history_tuned: history of the trained model
        fold: (int) index representing number of fold analyzed

    Es: plot_loss_figure(history,f)'''

    #plot
    plt.figure(figsize=(15,10))
    #plt.plot(history_tuned.history['accuracy'])
    #plt.plot(history_tuned.history['val_accuracy'])
    plt.plot(history_tuned['accuracy'])
    plt.plot(history_tuned['val_accuracy'])
    plt.title('Model Accuracy', fontsize = 50)
    plt.ylabel('Accuracy',fontsize=45)
    plt.yticks(np.arange(0, 1.2, step=0.1),fontsize = 35)
    plt.xlabel('Epoch',fontsize=45)
    plt.xticks(fontsize=35)
    plt.legend(['Train', 'Val'], loc='upper left', fontsize = 25)
    plt.savefig('densenet_acc_tuned[%d].png' % fold, dpi=600)
    plt.show()

    plt.figure(figsize=(15,10))
    #plt.plot(history_tuned.history['loss'])
    #plt.plot(history_tuned.history['val_loss'])
    plt.plot(history_tuned['loss'])
    plt.plot(history_tuned['val_loss'])
    plt.title('Model Loss', fontsize = 50)
    plt.ylabel('Loss', fontsize=45)
    plt.xlabel('Epoch', fontsize=45)
    plt.yticks(np.arange(0,1.2,0.1), fontsize=35)
    plt.xticks(fontsize=35)
    plt.ylim([0 ,1])
    plt.legend(['Train', 'Val'], loc='upper left', fontsize = 25)
    plt.savefig('densenet_loss_tuned[%d].png' % fold, dpi=600)
    plt.show()

predictions are obtained by averaging the predicted probabilities of rx belonging to the same patient'0s follow-up

In [ ]:
# 1. Predizioni dal modello
y_pred_probs = model.predict(x_val)

# 2. Indice della classe "failed"
fail_pos = 0

# 3. Etichette vere
y_true_labels = y_val[:, fail_pos].astype(int)

# 4. Aggiungi predizioni di probabilità al DataFrame
val_df['prob_failed'] = y_pred_probs[:, fail_pos]
val_df['true_label'] = y_true_labels

# 5. Estrai patient_id dal nome dell'immagine
val_df['patient_id'] = val_df['img'].str.extract(r'(O1-A-\d{4}-\d+m)')

# 6. Raggruppa per paziente e calcola la media delle probabilità
grouped_df = (
    val_df.groupby('patient_id')
    .agg({
        'true_label': 'first',           # assume stessa label per tutte le immagini
        'prob_failed': 'mean'            # media delle probabilità
    })
    .reset_index()
)

# 7. Applica soglia 0.5 per ottenere predizione binaria a livello paziente
grouped_df['predicted'] = (grouped_df['prob_failed'] >= 0.5).astype(int)

# Ora `grouped_df` contiene:
# - 'patient_id'
# - 'true_label'
# - 'prob_failed' (media delle probabilità)
# - 'predicted' (output finale binario)

print(grouped_df.head())


In [ ]:
from sklearn.metrics import confusion_matrix, balanced_accuracy_score, f1_score, recall_score, roc_auc_score

y_true = grouped_df['true_label']
y_pred = grouped_df['predicted']
y_prob = grouped_df['prob_failed']  # Probabilità media per AUC

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

# Metriche
bal_acc = balanced_accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
sensitivity = recall_score(y_true, y_pred)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
auc = roc_auc_score(y_true, y_prob)

print(f"Balanced Accuracy: {bal_acc:.3f}")
print(f"F1 Score: {f1:.3f}")
print(f"Sensitivity: {sensitivity:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"AUC: {auc:.3f}")


In [ ]:
import numpy as np
import pandas as pd

# 1. Predizioni dal modello
y_pred_probs = model.predict(x_test)

# 2. Indice della classe "failed"
fail_pos = 0

# 3. Etichette vere
y_true_labels = y_test[:, fail_pos].astype(int)

# 4. Aggiungi predizioni di probabilità al DataFrame
test_df['prob_failed'] = y_pred_probs[:, fail_pos]
test_df['true_label'] = y_true_labels

# 5. Estrai patient_id dal nome dell'immagine
test_df['patient_id'] = test_df['img'].str.extract(r'(O1-A-\d{4}-\d+m)')

# 6. Raggruppa per paziente e calcola la media delle probabilità
grouped_df = (
    test_df.groupby('patient_id')
    .agg({
        'true_label': 'first',           # assume stessa label per tutte le immagini
        'prob_failed': 'mean'            # media delle probabilità
    })
    .reset_index()
)

# 7. Applica soglia 0.5 per ottenere predizione binaria a livello paziente
grouped_df['predicted'] = (grouped_df['prob_failed'] >= 0.5).astype(int)

# Ora `grouped_df` contiene:
# - 'patient_id'
# - 'true_label'
# - 'prob_failed' (media delle probabilità)
# - 'predicted' (output finale binario)

print(grouped_df.head())


In [ ]:
from sklearn.metrics import confusion_matrix, balanced_accuracy_score, f1_score, recall_score, roc_auc_score

y_true = grouped_df['true_label']
y_pred = grouped_df['predicted']
y_prob = grouped_df['prob_failed']  # Probabilità media per AUC

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

# Metriche
bal_acc = balanced_accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
sensitivity = recall_score(y_true, y_pred)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
auc = roc_auc_score(y_true, y_prob)

print(f"Balanced Accuracy: {bal_acc:.3f}")
print(f"F1 Score: {f1:.3f}")
print(f"Sensitivity: {sensitivity:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"AUC: {auc:.3f}")


## Loading external validation data

In [ ]:
#@title Set the path
train_path=r''
output_path = r''
if not os.path.exists(output_path): #create folder if it doesn't exist
    os.makedirs(output_path)

train_failed=os.path.join(train_path,'classe 1')
train_notfal=os.path.join(train_path,'classe 0')
print(len(os.listdir(train_failed)))
print(len(os.listdir(train_notfal)))

In [ ]:
SEED = 4224
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

In [ ]:
import os
fal_names = []
fal_code = []
nonfal_names = []
nonfal_code = []

# Estensioni immagine ammesse
image_extensions = ('.jpg', '.jpeg', '.png')

# Cartella immagini fallite
for sample in os.listdir(train_failed):
    if sample.lower().endswith(image_extensions):  # solo immagini
        fal_code.append(sample.split('_')[0])  # estrae codice
        sample_path = os.path.join(train_failed, sample)
        fal_names.append(sample_path)

# Cartella immagini non fallite
for sample in os.listdir(train_notfal):
    if sample.lower().endswith(image_extensions):  # solo immagini
        nonfal_code.append(sample.split('_')[0])
        sample_path = os.path.join(train_notfal, sample)
        nonfal_names.append(sample_path)


### Adapting data to Densenet

In [ ]:
#@title Adapting image format to densenet
# We use densenet preprocess function in order to adapt the images to the input
# expected by the model
#carica immagine con formato 224,224 e prepara input
#mette insieme falliti e non falliti

from PIL import Image

data_img=[]


for i,name1 in enumerate(fal_names):
        es = load_img(name1,target_size=(224,224))
        x_ext=img_to_array(es)
        x_ext=tf.keras.applications.densenet.preprocess_input(x_ext)    #preparazione input specifico per densnet (siccome fa transfer learning)
        data_img.append(x_ext)

for i,nf_name1 in enumerate(nonfal_names):
        es=load_img(nf_name1,target_size=(224,224))
        x_ext=img_to_array(es)
        x_ext=tf.keras.applications.densenet.preprocess_input(x_ext)
        data_img.append(x_ext)


x_ext=np.array(data_img)

print(x_ext.shape)


In [ ]:
labels_df_ext=pd.DataFrame(columns=['img','label'])
i=0
for name1 in fal_names:
        labels_df_ext.loc[i]=pd.Series({'img': name1, 'label':'failed'})
        i=i+1
for nf_name1 in nonfal_names:
        labels_df_ext.loc[i]=pd.Series({'img': nf_name1, 'label':'not_failed'})
        i=i+1

In [ ]:
import re

def extract_patient_id_with_checkpoint_2(path):
    # Qui estrai O1-A-0128-12m dal path
    # Pattern: O1-A-<4 cifre>-<checkpoint>, esempio: O1-A-0128-12m
    match = re.search(r'(O2-A-\d{3}-\d+m)', path)
    if match:
        return match.group(1)
    else:
        return None  # o la vecchia patient_id se preferisci

val_df['patient_id'] = val_df['img'].apply(extract_patient_id_with_checkpoint)
train_df['patient_id'] = train_df['img'].apply(extract_patient_id_with_checkpoint)
test_df['patient_id'] = test_df['img'].apply(extract_patient_id_with_checkpoint)



labels_df_ext['patient_id'] = labels_df_ext['img'].apply(extract_patient_id_with_checkpoint_2)


In [ ]:
 #creazione variabile labels che contiene tante colonne quante sono le classi. (qui due colonne: failed/nfailed)
 #questo va bene con la softmax multiclass e catecorical cross-entropy che usiamo dopo, che ha tanti neuroni quante le classi
labels_train=labels_df_ext.label.to_list()
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(['failed','not_failed'])
train_labels = le.transform(labels_train)
y_ext=to_categorical(train_labels,2)
y_ext[:10],labels_df_ext[:10]

In [ ]:
#@title Set index failed
if y_ext[0,0] == 1:
  fail_pos = 0 #index where the failed label is positive
else:
  fail_pos = 1

### Predictions on external validation

In [ ]:
import numpy as np
import pandas as pd

# 1. Predizioni dal modello
y_pred_probs = model.predict(x_ext)

# 2. Indice della classe "failed"
fail_pos = 0

# 3. Etichette vere
y_true_labels = y_ext[:, fail_pos].astype(int)

# 4. Aggiungi predizioni di probabilità al DataFrame
labels_df_ext['prob_failed'] = y_pred_probs[:, fail_pos]
labels_df_ext['true_label'] = y_true_labels

# 5. Estrai patient_id dal nome dell'immagine
labels_df_ext['patient_id'] = labels_df_ext['img'].str.extract(r'(O2-A-\d{3}-\d+m)')

# 6. Raggruppa per paziente e calcola la media delle probabilità
grouped_df = (
    labels_df_ext.groupby('patient_id')
    .agg({
        'true_label': 'first',           # assume stessa label per tutte le immagini
        'prob_failed': 'mean'            # media delle probabilità
    })
    .reset_index()
)

# 7. Applica soglia 0.5 per ottenere predizione binaria a livello paziente
grouped_df['predicted'] = (grouped_df['prob_failed'] >= 0.5).astype(int)

# Ora `grouped_df` contiene:
# - 'patient_id'
# - 'true_label'
# - 'prob_failed' (media delle probabilità)
# - 'predicted' (output finale binario)

print(grouped_df.head())


In [ ]:
from sklearn.metrics import confusion_matrix, balanced_accuracy_score, f1_score, recall_score, roc_auc_score

y_true = grouped_df['true_label']
y_pred = grouped_df['predicted']
y_prob = grouped_df['prob_failed']  # Probabilità media per AUC

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

# Metriche
bal_acc = balanced_accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
sensitivity = recall_score(y_true, y_pred)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
auc = roc_auc_score(y_true, y_prob)

print(f"Balanced Accuracy: {bal_acc:.3f}")
print(f"F1 Score: {f1:.3f}")
print(f"Sensitivity: {sensitivity:.3f}")
print(f"Specificity: {specificity:.3f}")
print(f"AUC: {auc:.3f}")
